In [1]:
import tvm
from tvm import relax
from tvm.relax.frontend.torch import from_exported_program

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import torchvision.models as models
from torchvision.io import read_image
from torch.export import export

import onnx
import os

# Resnet model

In [ ]:
from torchvision.models.mobilenetv2 import MobileNet_V2_Weights, mobilenet_v2
torch_model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT).eval()

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /home/leobrasileo/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:02<00:00, 4.79MB/s]


IRModule

In [3]:
# Give an example argument to torch.export
example_args = (torch.randn(1, 3, 224, 224, dtype=torch.float32),)

# Convert the model to IRModule
with torch.no_grad():
    exported_program = export(torch_model, example_args)
    mod = from_exported_program(exported_program, keep_params_as_input=True)

mod, params = relax.frontend.detach_params(mod)
mod.show()

/usr/lib/python3/dist-packages/z3/z3core.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Optimize model

In [ ]:
TOTAL_TRIALS = 512
MAX_TRIALS_PER_TASK = 16 
target = tvm.target.Target({
    "kind": "llvm",
    "mtriple": "riscv64-linux-gnu",
    "mcpu": "generic-rv64",
    "mattr": ["+64bit", "+m", "+a", "+f", "+d", "+c", "+v"],
    "num-cores": 8,
})
work_dir = "tuning_logs"

mod = relax.get_pipeline(
    "static_shape_tuning",
    target=target,
    work_dir=work_dir,
    total_trials=TOTAL_TRIALS,
    max_trials_per_task=MAX_TRIALS_PER_TASK,
)(mod)

# Only show the main function
mod["main"].show()

2026-03-22 23:32:48 [INFO] Logging directory: tuning_logs/logs
2026-03-22 23:32:49 [INFO] LocalBuilder: max_workers = 8
2026-03-22 23:32:50 [INFO] LocalRunner: max_workers = 1
2026-03-22 23:32:52 [INFO] [task_scheduler.cc:171] Initializing Task #0: "fused_matmul_add13"
2026-03-22 23:32:52 [INFO] [task_scheduler.cc:171] Initializing Task #1: "transpose"
2026-03-22 23:32:52 [INFO] [task_scheduler.cc:171] Initializing Task #2: "reshape"
2026-03-22 23:32:52 [INFO] [task_scheduler.cc:171] Initializing Task #3: "mean"
2026-03-22 23:32:52 [INFO] [task_scheduler.cc:171] Initializing Task #4: "fused_conv2d_subtract_divide_expand_dims_multiply_expand_dims_add1_relu"
2026-03-22 23:32:52 [INFO] [task_scheduler.cc:171] Initializing Task #5: "fused_conv2d3_subtract2_divide2_expand_dims1_multiply2_expand_dims1_add5_add6_relu2"
2026-03-22 23:32:52 [INFO] [task_scheduler.cc:171] Initializing Task #6: "max_pool2d"
2026-03-22 23:32:52 [INFO] [task_scheduler.cc:171] Initializing Task #7: "fused_conv2d1_su

[23:32:52] /home/leobrasileo/Desktop/UBA/Tesis/tvm/src/s_tir/meta_schedule/schedule_rule/apply_custom_rule.cc:63: Warning: Unknown schedule rule "meta_schedule.pool_max" for target keys "["cpu"]". Checked ffi::Functions:
  s_tir.meta_schedule.cpu.meta_schedule.pool_max


2026-03-22 23:32:52 [INFO] [task_scheduler.cc:171] Initializing Task #11: "fused_conv2d6_subtract3_divide3_expand_dims2_multiply3_expand_dims2_add8_add9_relu3"
2026-03-22 23:32:52 [INFO] [task_scheduler.cc:171] Initializing Task #12: "fused_conv2d4_subtract2_divide2_expand_dims1_multiply2_expand_dims1_add5"
2026-03-22 23:32:52 [INFO] [task_scheduler.cc:171] Initializing Task #13: "fused_conv2d3_subtract2_divide2_expand_dims1_multiply2_expand_dims1_add5_relu2"
2026-03-22 23:32:52 [INFO] [task_scheduler.cc:171] Initializing Task #14: "fused_conv2d6_subtract3_divide3_expand_dims2_multiply3_expand_dims2_add8_relu3"
2026-03-22 23:32:52 [INFO] [task_scheduler.cc:171] Initializing Task #15: "fused_conv2d8_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_relu4"
2026-03-22 23:32:52 [INFO] [task_scheduler.cc:171] Initializing Task #16: "fused_conv2d9_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_add12_relu4"
2026-03-22 23:32:52 [INFO] [task_scheduler.cc:171] Initializing

,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,fused_matmul_add13,1025000,1,N/A,N/A,N/A,0,
1,transpose,1,1,N/A,N/A,N/A,0,
2,reshape,1,1,N/A,N/A,N/A,0,
3,mean,25600,1,N/A,N/A,N/A,0,
4,fused_conv2d_subtract_divide_expand_dims_multiply_expand_dims_add1_relu,240041984,1,N/A,N/A,N/A,0,
5,fused_conv2d3_subtract2_divide2_expand_dims1_multiply2_expand_dims1_add5_add6_relu2,231813120,2,N/A,N/A,N/A,0,
6,max_pool2d,1806336,1,N/A,N/A,N/A,0,
7,fused_conv2d1_subtract1_divide1_expand_dims_multiply1_expand_dims_add2_relu1,232214528,2,N/A,N/A,N/A,0,
8,fused_conv2d1_subtract1_divide1_expand_dims_multiply1_expand_dims_add2_add3_relu1,232415232,2,N/A,N/A,N/A,0,
9,fused_conv2d9_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_relu4,231336448,1,N/A,N/A,N/A,0,



Total trials: 0
Total latency (us): 0

2026-03-22 23:32:52 [DEBUG] [task_scheduler.cc:330] 
 ID |                                                                                  Name |      FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  0 |                                                                    fused_matmul_add13 |   1025000 |      1 |            N/A |          N/A |                   N/A |      0 |      
  1 |                                                                             transpose |         1 |      1 |            N/A |          N/A |                   N/A |      0 |      
  2 |                                                                               reshape |         1 |      1 |            N/A |          N/A |                 

[23:32:52] /home/leobrasileo/Desktop/UBA/Tesis/tvm/src/script/printer/tir/expr.cc:74: Warning: Didn't find variable definition for: v_k_o
[23:32:52] /home/leobrasileo/Desktop/UBA/Tesis/tvm/src/script/printer/tir/expr.cc:74: Warning: Didn't find variable definition for: v_k_o
[23:32:52] /home/leobrasileo/Desktop/UBA/Tesis/tvm/src/script/printer/tir/expr.cc:74: Warning: Didn't find variable definition for: v_k_o
[23:32:52] /home/leobrasileo/Desktop/UBA/Tesis/tvm/src/script/printer/tir/expr.cc:74: Warning: Didn't find variable definition for: v_k_o
[23:32:52] /home/leobrasileo/Desktop/UBA/Tesis/tvm/src/script/printer/tir/expr.cc:74: Warning: Didn't find variable definition for: v_k_o
[23:32:52] /home/leobrasileo/Desktop/UBA/Tesis/tvm/src/script/printer/tir/expr.cc:74: Warning: Didn't find variable definition for: v_k_o
[23:32:52] /home/leobrasileo/Desktop/UBA/Tesis/tvm/src/script/printer/ir/./../utils.h:47: Warning: TVMScript printer falls back to the basic address printer with the error:

2026-03-22 23:32:55 [INFO] [task_scheduler.cc:205] Sending 16 sample(s) to builder
2026-03-22 23:33:39 [INFO] [task_scheduler.cc:207] Sending 16 sample(s) to runner
2026-03-22 23:33:39 [INFO] [task_scheduler.cc:192] TaskScheduler picks Task #1: "transpose"
2026-03-22 23:33:40 [INFO] [task_scheduler.cc:205] Sending 1 sample(s) to builder
2026-03-22 23:33:49 [INFO] [task_scheduler.cc:207] Sending 1 sample(s) to runner
2026-03-22 23:33:49 [INFO] [task_scheduler.cc:192] TaskScheduler picks Task #2: "reshape"
2026-03-22 23:33:49 [INFO] [task_scheduler.cc:205] Sending 1 sample(s) to builder
2026-03-22 23:33:58 [INFO] [task_scheduler.cc:207] Sending 1 sample(s) to runner
2026-03-22 23:33:58 [INFO] [task_scheduler.cc:192] TaskScheduler picks Task #3: "mean"
2026-03-22 23:34:00 [INFO] [task_scheduler.cc:205] Sending 16 sample(s) to builder
2026-03-22 23:34:14 [INFO] [task_scheduler.cc:207] Sending 16 sample(s) to runner
2026-03-22 23:34:14 [INFO] [task_scheduler.cc:192] TaskScheduler picks Task

KeyboardInterrupt: 

Exception ignored in: 'tvm_ffi.core.tvm_ffi_callback'
Traceback (most recent call last):
  File "python/tvm_ffi/cython/function.pxi", line 75, in tvm_ffi.core.make_ret
  File "python/tvm_ffi/cython/object.pxi", line 408, in tvm_ffi.core.make_ret_object
  File "python/tvm_ffi/cython/string.pxi", line 64, in tvm_ffi.core.String.__from_tvm_ffi_object__
KeyboardInterrupt: 


2026-03-22 23:38:20 [INFO] [task_scheduler.cc:205] Sending 16 sample(s) to builder


In [ ]:
with target:
    mod = tvm.s_tir.transform.DefaultGPUSchedule()(mod)
ex = tvm.compile(mod, target=target)
dev = tvm.device("cuda", 0)
vm = relax.VirtualMachine(ex, dev)
# Need to allocate data and params on GPU device
gpu_data = tvm.runtime.tensor(np.random.rand(1, 3, 224, 224).astype("float32"), dev)
gpu_params = [tvm.runtime.tensor(p, dev) for p in params["main"]]
gpu_out = vm["main"](gpu_data, *gpu_params)[0].numpy()

print(gpu_out.shape)